In [ ]:
import sys
print("Python:", sys.version)

import pandas as pd
import networkx as nx
print("Pandas:", pd.__version__)
print("NetworkX:", nx.__version__)

print("VS Code Jupyter setup working! 🎉")

Python: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
Pandas: 3.0.1
NetworkX: 3.6.1
VS Code Jupyter7 setup working! 🎉


In [13]:
from pathlib import Path

files = [
    Path("../data/english_4.txt"),
    Path("../data/english_5.txt"),
]

for path in files:
    if not path.exists():
        print(f"Missing: {path}")
        continue

    words = [line.strip() for line in path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
    unique_words = set(words)

    print(f"{path} -> total lines: {len(words)}, unique words: {len(unique_words)}")
    print(f"first 10: {words[:10]}")
    print()

..\data\english_4.txt -> total lines: 3382, unique words: 3382
first 10: ['aahs', 'abba', 'abbe', 'abbr', 'abed', 'abet', 'able', 'ably', 'abut', 'acct']

..\data\english_5.txt -> total lines: 11155, unique words: 11155
first 10: ['aahed', 'aalii', 'aargh', 'abaca', 'abaci', 'aback', 'abaft', 'abaka', 'abamp', 'aband']



In [14]:
from pathlib import Path
from collections import Counter
from urllib.request import urlopen

base = Path("../data")
raw_dir = base / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Final outputs
out_5 = base / "english_5.txt"                # non-strict (big)
out_5_strict = base / "english_5_strict.txt"  # strict (~5800)
out_4 = base / "english_4.txt"                # non-strict
out_4_strict = base / "english_4_strict.txt"  # strict

# External sources
wordle_valid_url = "https://gist.githubusercontent.com/dracos/dd0668f281e685bad51479e5acaadb93/raw/valid-wordle-words.txt"
wordle_valid_file = raw_dir / "valid-wordle-words.txt"

paulcc_4_url = "https://gist.githubusercontent.com/paulcc/3799331/raw/"
paulcc_4_file = raw_dir / "paulcc_4letter_words.txt"

rpi_4_url = "https://gist.githubusercontent.com/raspberrypisig/cc18b0f4fbc0c79ffd667d06adc0a190/raw/4-letter-words-processed-new.txt"
rpi_4_file = raw_dir / "rpi_4letter_words.txt"

rmmh_4_url = "https://raw.githubusercontent.com/rmmh/fourletterword/master/list.txt"
rmmh_4_file = raw_dir / "rmmh_4letter_words.txt"

for url, path in [
    (wordle_valid_url, wordle_valid_file),
    (paulcc_4_url, paulcc_4_file),
    (rpi_4_url, rpi_4_file),
    (rmmh_4_url, rmmh_4_file),
]:
    if not path.exists():
        path.write_text(
            urlopen(url, timeout=60).read().decode("utf-8", errors="ignore"),
            encoding="utf-8",
        )

source_files = {
    "sgb": raw_dir / "sgb-words.txt",          # Donald Knuth base (5-letter)
    "dwyl": raw_dir / "dwyl_words.txt",
    "doublet": raw_dir / "doublet_words.txt",
    "knuth_alt": raw_dir / "knuth_words.txt",
    "wordl3": raw_dir / "wordl3.txt",          # optional, may be missing
    "wordle_valid": wordle_valid_file,
    "paulcc_4": paulcc_4_file,
    "rpi_4": rpi_4_file,
    "rmmh_4": rmmh_4_file,
}

def load_set(path: Path, n: int) -> set[str]:
    if not path.exists():
        return set()
    words = {
        line.strip().lower()
        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines()
    }
    return {w for w in words if len(w) == n and w.isascii() and w.isalpha()}

# 5-letter strict policy (~5800 set):
# - Knuth base + words present in >=2 non-SGB sources
base_5 = load_set(source_files["sgb"], 5)
other_for_5 = ["dwyl", "doublet", "knuth_alt", "wordl3"]
other_sets_5 = [
    load_set(source_files[name], 5)
    for name in other_for_5
    if source_files[name].exists()
]
count_5 = Counter(w for s in other_sets_5 for w in s)
strict_add_5 = {w for w, c in count_5.items() if c >= 2}
final_5_strict = sorted(base_5 | strict_add_5)

# 5-letter non-strict policy (big):
# - Knuth base + (word in valid Wordle guesses and in >=1 non-SGB source)
wordle_valid_5 = load_set(source_files["wordle_valid"], 5)
non_strict_add_5 = {w for w in wordle_valid_5 if count_5.get(w, 0) >= 1}
final_5 = sorted(base_5 | non_strict_add_5)

# 4-letter source groups
core_4_sets = [
    load_set(source_files["dwyl"], 4),
    load_set(source_files["doublet"], 4),
    load_set(source_files["knuth_alt"], 4),
    load_set(source_files["wordl3"], 4) if source_files["wordl3"].exists() else set(),
]

curated_4_sets = [
    load_set(source_files["doublet"], 4),
    load_set(source_files["paulcc_4"], 4),
    load_set(source_files["rpi_4"], 4),
    load_set(source_files["rmmh_4"], 4),
]

# 4-letter strict policy (expanded but still filtered):
# - word must appear in >=2 curated 4-letter sources
count_4_strict = Counter(w for s in curated_4_sets for w in s)
final_4_strict = sorted({w for w, c in count_4_strict.items() if c >= 2})

# 4-letter non-strict policy:
# - strict set + full union of curated 4-letter sources
final_4 = sorted(set(final_4_strict).union(*curated_4_sets))

# Write all outputs
out_5.write_text("\n".join(final_5) + "\n", encoding="utf-8")
out_5_strict.write_text("\n".join(final_5_strict) + "\n", encoding="utf-8")
out_4.write_text("\n".join(final_4) + ("\n" if final_4 else ""), encoding="utf-8")
out_4_strict.write_text("\n".join(final_4_strict) + ("\n" if final_4_strict else ""), encoding="utf-8")

print("Policy summary:")
print("- english_5.txt: big non-strict")
print("- english_5_strict.txt: Knuth + >=2 non-SGB sources")
print("- english_4_strict.txt: >=2 curated 4-letter sources")
print("- english_4.txt: union of curated 4-letter sources")
print()

print("Counts:")
print(f"- english_5.txt: {len(final_5)}")
print(f"- english_5_strict.txt: {len(final_5_strict)}")
print(f"- english_4.txt: {len(final_4)}")
print(f"- english_4_strict.txt: {len(final_4_strict)}")
print()

print("4-letter source sizes:")
print(f"- doublet_4: {len(curated_4_sets[0])}")
print(f"- paulcc_4: {len(curated_4_sets[1])}")
print(f"- raspberrypisig_4: {len(curated_4_sets[2])}")
print(f"- rmmh_4: {len(curated_4_sets[3])}")
print(f"- added_to_non_strict_4: {len(set(final_4) - set(final_4_strict))}")

Policy summary:
- english_5.txt: big non-strict
- english_5_strict.txt: Knuth + >=2 non-SGB sources
- english_4_strict.txt: >=2 curated 4-letter sources
- english_4.txt: union of curated 4-letter sources

Counts:
- english_5.txt: 11155
- english_5_strict.txt: 5811
- english_4.txt: 5733
- english_4_strict.txt: 3176

4-letter source sizes:
- doublet_4: 1353
- paulcc_4: 3130
- raspberrypisig_4: 2252
- rmmh_4: 5526
- added_to_non_strict_4: 2557
